# SpectraAE: 1D Conv Autoencoder + PU-Bagging CN Star Detection

**Core Goal**: Use unsupervised 1D convolutional autoencoders to compress LAMOST spectra (700px -> 64/256 dim),
extract bottleneck features, then classify CN-enhanced stars with PU-Bagging (XGBoost).

**CN Molecular Bands**: CN3839 (3830-3883A), CN4142 (4120-4216A), CH4300 (4285-4315A)

**Key Question**: Raw Spectra 700px + PU-Bagging achieves PR-AUC=0.848.
Can AE-compressed features retain CN band discriminative information?

---
## Three AE Variants

| Variant | Bottleneck | Params | Compression | Feature |
|---------|-----------|--------|-------------|---------|
| **Vanilla AE-64d** | 64 | 87,296 | 10.9x | Standard MSE loss, baseline |
| **AE-256d** | 256 | ~500K | 2.7x | Larger capacity, more continuum detail |
| **CN-Aware AE-64d** | 64 | 87,296 | 10.9x | 5x weighted MSE on CN band regions |

**Core Findings**:
- Standard AE features achieve PR-AUC ~0.17, far below raw spectra 0.85
- CN-Aware training gives ~8.4% relative improvement (PR 0.181 vs 0.167)
- 256d vs 64d yields marginal gain (PR 0.173 vs 0.167), extra capacity captures continuum


In [ ]:
import sys, warnings, pickle, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 11, 'figure.dpi': 120})
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

_PROJECT_ROOT = Path().resolve().parent
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

from SpectraAE.models.autoencoder import ConvAutoencoder
from SpectraAE.extract_features import extract_features
from SpectraAE.cn_aware_pretrain import create_band_weight_mask, CN_BAND_DEFS
from ML.pu_bagging import load_pu_data, run_pu_bagging, build_comparison_df

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')


## 1. Data Overview

Loading preprocessed 33,589 LAMOST spectra from ML/_cache/X_clean.npy.
Each spectrum: 700 pixels (3800-4500A, 1A step), continuum-normalized (flux ~1.0).

**Labels**: 73 known CN-enhanced positives (from CNstar.csv + FT_cands.csv), 33,516 unlabeled.


In [ ]:
# --- Load Data ---
CACHE_DIR = Path('ML/_cache')
X_clean = np.load(CACHE_DIR / 'X_clean.npy').astype(np.float32)
stars_clean = pd.read_pickle(CACHE_DIR / 'stars_clustered.pkl')
wave = np.arange(3800.0, 4500.0, 1.0)

# Load PU-Bagging labels/splits
data_pu = load_pu_data()
y_all = data_pu['y_all']
cluster_ids = data_pu['cluster_ids']
df_model = data_pu['df_model']
pos_mask = y_all == 1
n_pos = int(y_all.sum())

print(f'Spectra shape: {X_clean.shape} (N={X_clean.shape[0]:,}, pixels={X_clean.shape[1]})')
print(f'Pixel range: [{X_clean.min():.4f}, {X_clean.max():.4f}]')
print(f'Global stats: mean={X_clean.mean():.4f}, std={X_clean.std():.4f}')
print(f'Labels: {len(y_all):,} stars, {n_pos} known CN + {(~pos_mask).sum():,} unlabeled')


### 1.1 Spectrum Preview: CN Stars vs Normal Stars

The three CN/CH bands are highlighted. CN-enhanced stars show deeper absorption in these regions.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# --- Left: Mean spectra comparison ---
ax = axes[0]
ax.plot(wave, X_clean[~pos_mask].mean(axis=0), alpha=0.8, lw=1,
        color='#3498db', label=f'Unlabeled (n={(~pos_mask).sum():,})')
ax.plot(wave, X_clean[pos_mask].mean(axis=0), alpha=0.9, lw=1.5,
        color='#e74c3c', label=f'Known CN (n={n_pos})')
# Annotate molecular bands
band_styles = [('CN3839', (3830, 3883), '#3498db'),
               ('CN4142', (4120, 4216), '#2ecc71'),
               ('CH4300', (4285, 4315), '#e67e22')]
for name, (x1, x2), color in band_styles:
    ax.axvspan(x1, x2, alpha=0.1, color=color, label=name)
ax.set_xlabel('Wavelength (A)')
ax.set_ylabel('Normalized Flux')
ax.set_title('Mean Spectra: CN vs Normal')
ax.legend(fontsize=8, loc='lower right')
ax.grid(alpha=0.2)

# --- Right: Difference spectrum (Normal - CN) ---
ax = axes[1]
diff = X_clean[~pos_mask].mean(axis=0) - X_clean[pos_mask].mean(axis=0)
ax.plot(wave, diff, lw=1.2, color='#8e44ad')
ax.fill_between(wave, 0, diff, alpha=0.3, color='#8e44ad')
for _, (x1, x2), color in band_styles:
    ax.axvspan(x1, x2, alpha=0.1, color=color)
ax.axhline(y=0, color='black', lw=0.5)
ax.set_xlabel('Wavelength (A)')
ax.set_ylabel('Flux Difference')
ax.set_title('Mean Difference: Normal - CN (CN absorption excess)')
ax.grid(alpha=0.2)

fig.suptitle('LAMOST Spectra Overview - CN Band Identification', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


## 2. Autoencoder Model Architecture

**Encoder**: 3-layer 1D Conv (stride=2) downsampling + AdaptiveAvgPool1d + FC -> bottleneck
**Decoder**: FC + 4-stage linear interpolation upsampling + 1D Conv -> 700px reconstruction

Three variants parameter comparison:


In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters())

def model_summary(name, base_ch, latent_dim):
    m = ConvAutoencoder(in_channels=1, base_ch=base_ch, latent_dim=latent_dim)
    n = count_params(m)
    comp = 700 / latent_dim
    print(f'{name:20s} | base_ch={base_ch:2d} latent={latent_dim:3d} | '
          f'{n:>8,} params | Compression 700->{latent_dim} ({comp:.1f}x)')
    return m

print('=' * 72)
print(f'{"Model":20s} | {"Config":18s} | {"Params":>10s} | Compression')
print('=' * 72)
_ = model_summary('Vanilla AE-64d', 32, 64)
_ = model_summary('AE-256d', 64, 256)
_ = model_summary('CN-Aware AE-64d', 32, 64)
print('=' * 72)


### 2.1 Encoder Shape Progression

Trace the tensor shape through each layer of the encoder.


In [ ]:
def trace_encoder(enc, x_dummy, name):
    print(f'\n{name}:')
    shapes = [('Input', tuple(x_dummy.shape))]
    x = enc.pad(x_dummy)
    shapes.append(('ReflectionPad1d(2)', tuple(x.shape)))
    for i, conv in enumerate([enc.conv1, enc.conv2, enc.conv3], 1):
        x = conv(x)
        shapes.append((f'Conv1d-{i} (stride=2)', tuple(x.shape)))
    x = enc.pool(x)
    shapes.append(('AdaptiveAvgPool1d(1)', tuple(x.shape)))
    x = x.flatten(1)
    shapes.append(('Flatten', tuple(x.shape)))
    z = enc.fc(x)
    shapes.append(('FC -> Latent z', tuple(z.shape)))
    for label, s in shapes:
        print(f'  {label:25s} {str(s):20s}')

ae_demo = ConvAutoencoder(in_channels=1, base_ch=32, latent_dim=64)
trace_encoder(ae_demo.encoder, torch.randn(1, 1, 700), 'Vanilla AE-64d / CN-Aware AE-64d')
ae256_demo = ConvAutoencoder(in_channels=1, base_ch=64, latent_dim=256)
trace_encoder(ae256_demo.encoder, torch.randn(1, 1, 700), 'AE-256d')


## 3. Pretraining: Unsupervised Spectrum Reconstruction

### Training Configuration
- **Loss**: MSE (per-pixel reconstruction error); CN-Aware uses weighted MSE
- **Optimizer**: AdamW (lr=1e-3, weight_decay=1e-5)
- **LR Scheduler**: CosineAnnealingLR (T_max=n_epochs, eta_min=1e-6)
- **Early Stopping**: patience=30 epochs (vanilla/CN-aware) / 40 epochs (256d)
- **Preprocessing**: Global p1-p99 clipping + scalar standardization
- **Data split**: 90% train / 10% validation
- **Hardware**: RTX 4060 Laptop GPU

### CN-Aware Weighted MSE Principle
Three CN/CH molecular band regions (182 pixels total, 26%) receive 5x weight,
forcing the AE to prioritize preserving details in these regions:


In [ ]:
# Visualize CN-Aware weight mask
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Weight mask at different multipliers
ax = axes[0]
for bw, ls, alpha in [(1.0, '-', 0.3), (3.0, '--', 0.6), (5.0, '-', 1.0), (8.0, ':', 0.7)]:
    wm = create_band_weight_mask(700, band_weight=bw).squeeze().numpy()
    ax.plot(wave, wm, lw=1.5, linestyle=ls, alpha=alpha, label=f'band_weight={bw}x')
ax.set_xlabel('Wavelength (A)')
ax.set_ylabel('Pixel Weight')
ax.set_title('CN-Aware Weight Masks at Different Multipliers')
ax.legend()
ax.grid(alpha=0.2)

# Pie chart of pixel counts
ax = axes[1]
wm_default = create_band_weight_mask(700, band_weight=5.0).squeeze().numpy()
n_band = int(np.sum(wm_default > 1.1))
n_cont = 700 - n_band
total_weight = np.sum(wm_default)
band_contrib = np.sum(wm_default[wm_default > 1.1]) / total_weight * 100
sizes = [n_cont, n_band]
labels = [f'Continuum ({n_cont}px)\nweight=1.0',
          f'CN Bands ({n_band}px)\nweight=5.0']
colors = ['#bdc3c7', '#e74c3c']
ax.pie(sizes, labels=labels, colors=colors, startangle=90, explode=(0, 0.05))
ax.set_title('Pixel Count: Band vs Continuum')

fig.suptitle('CN-Aware Weight Mask Design (band_weight=5.0)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print(f'Band pixels: {n_band} ({n_band/700*100:.1f}%), '
      f'Continuum pixels: {n_cont} ({n_cont/700*100:.1f}%)')
print(f'Weight contribution: Band={band_contrib:.1f}%, Continuum={cont_contrib:.1f}%')
print(f'Bands: {list(CN_BAND_DEFS.keys())}')


### 3.1 Load Trained Checkpoints

In [ ]:
def load_ae_checkpoint(ckpt_path, latent_dim, base_ch=32):
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    model = ConvAutoencoder(in_channels=1, base_ch=base_ch, latent_dim=latent_dim).to(DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    info = {
        'epoch': ckpt['epoch'], 'val_loss': ckpt['val_loss'],
        'scaler_mean': ckpt['scaler_mean'], 'scaler_std': ckpt['scaler_std'],
        'params': sum(p.numel() for p in model.parameters()),
    }
    return model, info

# Load three AE variants
checkpoints = {}

# Vanilla AE-64d
ckpt_path = Path('SpectraAE/checkpoints/ae_best.pt')
if ckpt_path.exists():
    model64, info64 = load_ae_checkpoint(ckpt_path, latent_dim=64, base_ch=32)
    checkpoints['Vanilla AE-64d'] = (model64, info64)
    print(f'Vanilla AE-64d:  epoch={info64["epoch"]}  val_loss={info64["val_loss"]:.6f}  '
          f'params={info64["params"]:,}')
else:
    print('Vanilla AE-64d checkpoint not found')

# AE-256d
ckpt256_path = Path('SpectraAE/checkpoints/ae256/ae_best.pt')
if ckpt256_path.exists():
    model256, info256 = load_ae_checkpoint(ckpt256_path, latent_dim=256, base_ch=64)
    checkpoints['AE-256d'] = (model256, info256)
    print(f'AE-256d:         epoch={info256["epoch"]}  val_loss={info256["val_loss"]:.6f}  '
          f'params={info256["params"]:,}')
else:
    print('AE-256d checkpoint not found')

# CN-Aware AE-64d
ckpt_cn_path = Path('SpectraAE/checkpoints/cn_aware/ae_best.pt')
if ckpt_cn_path.exists():
    model_cn, info_cn = load_ae_checkpoint(ckpt_cn_path, latent_dim=64, base_ch=32)
    checkpoints['CN-Aware AE-64d'] = (model_cn, info_cn)
    print(f'CN-Aware AE-64d: epoch={info_cn["epoch"]}  val_loss={info_cn["val_loss"]:.6f}  '
          f'params={info_cn["params"]:,}')
else:
    print('CN-Aware AE-64d checkpoint not found')


### 3.2 Training Curves

Compare AE-256d and CN-Aware AE-64d training processes.
CN-Aware version additionally tracks band and continuum region losses separately.


In [ ]:
# Load training histories
ae256_hist_path = Path('SpectraAE/_cache/ae256_history.pkl')
cn_hist_path = Path('SpectraAE/_cache/cn_aware_history.pkl')

has_vanilla_hist = ae256_hist_path.exists()
has_cn_hist = cn_hist_path.exists()

if has_cn_hist or has_vanilla_hist:
    n_plots = (1 if has_vanilla_hist else 0) + (1 if has_cn_hist else 0)
    fig, axes = plt.subplots(1, n_plots, figsize=(7 * n_plots, 5))
    if n_plots == 1:
        axes = [axes]

    plot_idx = 0

    if has_vanilla_hist:
        with open(ae256_hist_path, 'rb') as f:
            hist = pickle.load(f)
        ax = axes[plot_idx]
        epochs = range(1, len(hist['train_losses']) + 1)
        ax.plot(epochs, hist['train_losses'], lw=1, alpha=0.7, color='#3498db', label='Train')
        ax.plot(epochs, hist['val_losses'], lw=1.5, color='#e74c3c', label='Val')
        ax.axvline(x=hist['best_epoch'], color='green', linestyle='--', alpha=0.6,
                  label=f'Best epoch={hist["best_epoch"]}')
        ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
        ax.set_title(f'AE-256d Training (best_val={hist["best_val_loss"]:.6f}, '
                     f'{hist["elapsed_seconds"]:.0f}s)')
        ax.legend(); ax.grid(alpha=0.2)
        plot_idx += 1

    if has_cn_hist:
        with open(cn_hist_path, 'rb') as f:
            hist_cn = pickle.load(f)
        ax = axes[plot_idx]
        epochs_cn = range(1, len(hist_cn['train_losses']) + 1)
        ax.plot(epochs_cn, hist_cn['val_losses'], lw=2, color='#2c3e50', label='Val Total (weighted)')
        ax.plot(epochs_cn, hist_cn['val_band_losses'], lw=1.2, color='#e74c3c',
                label='Val Band (CN/CH regions)')
        ax.plot(epochs_cn, hist_cn['val_cont_losses'], lw=1.2, color='#3498db',
                label='Val Continuum')
        ax.axvline(x=hist_cn['best_epoch'], color='green', linestyle='--', alpha=0.6,
                  label=f'Best epoch={hist_cn["best_epoch"]}')
        ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
        ax.set_title(f'CN-Aware AE-64d (band_weight={hist_cn["band_weight"]}x, '
                     f'{hist_cn["elapsed_seconds"]:.0f}s)')
        ax.legend(fontsize=8); ax.grid(alpha=0.2)
        plot_idx += 1

    fig.suptitle('AE Training Curves', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('Training history files not found')


### 3.3 Reconstruction Quality Comparison

Randomly selected spectra, compare original vs. reconstruction from three AE variants.


In [ ]:
def prepare_for_ae(X, scaler_mean, scaler_std):
    lo = float(np.percentile(X, 1))
    hi = float(np.percentile(X, 99))
    X_c = np.clip(X, lo, hi)
    return (X_c - scaler_mean) / scaler_std

def reconstruct_batch(model, X, sm, ss):
    X_norm = prepare_for_ae(X, sm, ss)
    X_t = torch.from_numpy(X_norm).unsqueeze(1).to(DEVICE)
    with torch.no_grad():
        recon_t, _ = model(X_t)
    return recon_t.cpu().numpy()[:, 0, :], X_norm

# Randomly select 8 spectra
rng = np.random.RandomState(42)
n_show = 8
sample_idx = rng.choice(len(X_clean), n_show, replace=False)
X_sample = X_clean[sample_idx]

available_models = [(name, model, info) for name, (model, info) in checkpoints.items()]
n_models = len(available_models)

if n_models > 0:
    fig, axes = plt.subplots(n_show, 1 + n_models, figsize=(4 * (1 + n_models), 2.5 * n_show))
    if n_show == 1:
        axes = axes.reshape(1, -1)

    color_map = {'Vanilla AE-64d': '#e74c3c', 'AE-256d': '#2ecc71', 'CN-Aware AE-64d': '#f39c12'}

    for i in range(n_show):
        ax = axes[i, 0]
        X_norm_ref = prepare_for_ae(X_sample[i:i+1], info64['scaler_mean'], info64['scaler_std'])[0]
        ax.plot(wave, X_norm_ref, lw=0.8, color='#2c3e50')
        if i == 0:
            ax.set_title('Original (normalized)')
        ax.set_ylabel(f'#{sample_idx[i]}', fontsize=8)
        ax.grid(alpha=0.15)

        for j, (name, model, info) in enumerate(available_models):
            ax = axes[i, 1 + j]
            recon, X_norm = reconstruct_batch(model, X_sample[i:i+1],
                                              info['scaler_mean'], info['scaler_std'])
            ax.plot(wave, X_norm[0], lw=0.3, alpha=0.35, color='gray')
            ax.plot(wave, recon[0], lw=1, color=color_map.get(name, '#e74c3c'))
            mse = np.mean((X_norm[0] - recon[0])**2)
            if i == 0:
                ax.set_title(f'{name}\nMSE={mse:.4f}', fontsize=9,
                            color=color_map.get(name, '#e74c3c'))
            else:
                ax.set_title(f'MSE={mse:.4f}', fontsize=8,
                            color=color_map.get(name, '#e74c3c'))
            ax.grid(alpha=0.15)

    fig.suptitle('Reconstruction Quality: All AE Variants', fontsize=13, y=1.005)
    plt.tight_layout()
    plt.show()
else:
    print('No trained checkpoints available')


### 3.4 Per-Pixel Reconstruction Error

Analyze MSE per wavelength pixel, focusing on CN molecular band regions.
CN-Aware AE should have relatively lower error in band regions.


In [ ]:
def compute_per_pixel_mse(model, X, sm, ss, n_sample=5000):
    idx = np.random.RandomState(42).choice(len(X), min(n_sample, len(X)), replace=False)
    recon, X_norm = reconstruct_batch(model, X[idx], sm, ss)
    return np.mean((X_norm - recon)**2, axis=0)

pixel_mse = {}
for name, model, info in checkpoints.items():
    pixel_mse[name] = compute_per_pixel_mse(model, X_clean,
                                             info['scaler_mean'], info['scaler_std'])
    print(f'{name:20s} Global MSE={np.mean(pixel_mse[name]):.6f}')
    print(f'  CN3839 (3830-3883A): {np.mean(pixel_mse[name][30:83]):.6f}')
    print(f'  CN4142 (4120-4216A): {np.mean(pixel_mse[name][320:416]):.6f}')
    print(f'  CH4300 (4285-4315A): {np.mean(pixel_mse[name][485:515]):.6f}')
    print(f'  Continuum (4200-4280A): {np.mean(pixel_mse[name][400:480]):.6f}')

if len(pixel_mse) > 0:
    fig, ax = plt.subplots(figsize=(14, 5))
    pcolor_map = {'Vanilla AE-64d': '#e74c3c', 'AE-256d': '#2ecc71', 'CN-Aware AE-64d': '#f39c12'}
    for name, mse in pixel_mse.items():
        ax.plot(wave, mse, lw=1.5, color=pcolor_map.get(name, '#95a5a6'), label=name)

    for band_name, (x1, x2), color in band_styles:
        ax.axvspan(x1, x2, alpha=0.08, color=color)
        y_max = ax.get_ylim()[1]
        ax.text((x1+x2)/2, y_max * 0.92, band_name, ha='center', fontsize=8, color=color)

    ax.set_xlabel('Wavelength (A)')
    ax.set_ylabel('MSE per pixel')
    ax.set_title('Per-Pixel Reconstruction Error: All AE Variants')
    ax.legend()
    ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()


## 4. Bottleneck Feature Extraction

Use trained Encoders to compress all 33,589 spectra into bottleneck vectors,
serving as input features for downstream PU-Bagging classification.


In [ ]:
# Load or extract features
feature_cache = {
    'Vanilla AE-64d': ('SpectraAE/_cache/ae_features_64d.npy', model64, info64),
    'AE-256d': ('SpectraAE/_cache/ae_features_256d.npy', model256, info256),
    'CN-Aware AE-64d': ('SpectraAE/_cache/ae_features_cn_64d.npy', model_cn, info_cn),
}

ae_features = {}
for name, (cache_path, model, info) in feature_cache.items():
    if name not in checkpoints:
        continue
    fp = Path(cache_path)
    if fp.exists():
        feats = np.load(fp).astype(np.float32)
        print(f'{name:20s}: loaded {feats.shape} from cache')
    else:
        feats = extract_features(model, X_clean,
            scaler_mean=info['scaler_mean'], scaler_std=info['scaler_std'], device=DEVICE)
        np.save(fp, feats)
        print(f'{name:20s}: extracted {feats.shape} and saved')
    ae_features[name] = feats
    print(f'  stats: mean={feats.mean():.4f}  std={feats.std():.4f}  '
          f'min={feats.min():.4f}  max={feats.max():.4f}')


### 4.1 Latent Space PCA Visualization

Project bottleneck features to 2D PCA space, observe known CN star distribution
and Teff temperature gradient structure.


In [ ]:
n_feat = len(ae_features)
if n_feat > 0:
    fig, axes = plt.subplots(1, n_feat, figsize=(6 * n_feat, 5.5))
    if n_feat == 1:
        axes = [axes]

    for ax, (name, feats) in zip(axes, ae_features.items()):
        pca = PCA(n_components=2).fit(feats)
        z_pca = pca.transform(feats)

        unlabeled_idx = np.where(~pos_mask)[0]
        bg = np.random.RandomState(42).choice(unlabeled_idx, min(5000, len(unlabeled_idx)), replace=False)
        teff_bg = df_model.iloc[bg]['teff'].values
        sc = ax.scatter(z_pca[bg, 0], z_pca[bg, 1], s=1, alpha=0.4, c=teff_bg,
                       cmap='RdYlBu_r', edgecolors='none')
        plt.colorbar(sc, ax=ax, label='Teff (K)', shrink=0.85)

        pos_in_bg = bg[np.isin(bg, np.where(pos_mask)[0])]
        ax.scatter(z_pca[pos_in_bg, 0], z_pca[pos_in_bg, 1], s=50, alpha=0.9,
                  c='black', edgecolors='white', linewidth=0.5, marker='*', label='Known CN')

        ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
        ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
        ax.set_title(f'{name} Latent Space (PCA)')
        ax.legend(fontsize=8, loc='lower left')
        ax.grid(alpha=0.15)

    fig.suptitle('Bottleneck Feature Space - PCA Projection Colored by Teff', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('No features available')


### 4.2 Bottleneck Dimension Analysis: CN vs Unlabeled

Per-dimension comparison of feature distributions for known CN vs. unlabeled stars,
identifying which dimensions capture CN-discriminative signal.


In [ ]:
if 'Vanilla AE-64d' in ae_features and 'CN-Aware AE-64d' in ae_features:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    for ax, (name, feats) in zip(axes, [
        ('Vanilla AE-64d', ae_features['Vanilla AE-64d']),
        ('CN-Aware AE-64d', ae_features['CN-Aware AE-64d']),
    ]):
        cn_mean = feats[pos_mask].mean(axis=0)
        unl_mean = feats[~pos_mask].mean(axis=0)
        cn_std = feats[pos_mask].std(axis=0)
        unl_std = feats[~pos_mask].std(axis=0)
        pooled_std = np.sqrt((cn_std**2 + unl_std**2) / 2)
        cohens_d = np.abs(cn_mean - unl_mean) / (pooled_std + 1e-8)

        sorted_idx = np.argsort(cohens_d)[::-1]
        colors = ['#e74c3c' if d > 0.5 else '#f39c12' if d > 0.2 else '#bdc3c7'
                 for d in cohens_d[sorted_idx]]
        ax.bar(range(len(cohens_d)), cohens_d[sorted_idx], color=colors, edgecolor='white', width=0.8)
        ax.axhline(y=0.5, color='#e74c3c', linestyle='--', alpha=0.5, label='|d|=0.5 (medium)')
        ax.axhline(y=0.2, color='#f39c12', linestyle='--', alpha=0.5, label='|d|=0.2 (small)')
        ax.set_xlabel('Dimension (sorted by |d|)')
        ax.set_ylabel("Cohen's |d| (CN vs Unlabeled)")
        ax.set_title(f'{name}: Per-Dimension CN Separability')
        ax.legend(fontsize=7)
        ax.grid(axis='y', alpha=0.2)

        n_medium = int(np.sum(cohens_d > 0.5))
        n_small = int(np.sum(cohens_d > 0.2))
        print(f'{name}: |d|>0.5 = {n_medium}/64 dims  |d|>0.2 = {n_small}/64 dims')

    fig.suptitle('Bottleneck Dimension CN Separability Comparison', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()


## 5. PU-Bagging Classification Evaluation

Under the same data split, run PU-Bagging (T=500 iterations) on three AE feature sets
and raw spectra, with comprehensive performance comparison.

### Evaluation Metrics
| Metric | Meaning | Direction |
|--------|---------|-----------|
| **PR-AUC** | Precision-Recall AUC, primary ranking metric | Higher is better |
| **ROC-AUC** | ROC curve AUC | Higher is better |
| **P@100** | Precision in top-100 candidates | Higher is better |
| **Within-r** | Within-cluster Spearman correlation with delta_CN3839, physical consistency | Higher is better |
| **|r_teff| z** | De-biased Teff parameter correlation z-score | Lower is better |
| **Mean bias z** | Mean probability bias z-score | Lower is better |


In [ ]:
result_files = {
    'AE-64d vs Spectra': 'SpectraAE/results/pu_bagging_ae_comparison.csv',
    'AE-256d vs AE-64d vs Spectra': 'SpectraAE/results/pu_bagging_ae256_comparison.csv',
    'CN-Aware vs Vanilla vs Spectra': 'SpectraAE/results/pu_bagging_cn_aware_comparison.csv',
}

all_comparisons = {}
for label, path in result_files.items():
    fp = Path(path)
    if fp.exists():
        df = pd.read_csv(fp)
        all_comparisons[label] = df
        print(f'Loaded {label}: {df.shape[0]} rows')
    else:
        print(f'Not found: {path}')

all_results = []
seen = set()
for df in all_comparisons.values():
    for _, row in df.iterrows():
        key = row['Feature Set']
        if key not in seen:
            seen.add(key)
            all_results.append(row)

if all_results:
    summary_df = pd.DataFrame(all_results).sort_values('PR', ascending=False)

    display_cols = ['Feature Set', 'Dim', 'ROC', 'PR', 'P@50', 'P@100',
                    '|r_teff| z', 'Mean bias z', 'Within-r', 'Stability', 'Time']
    disp = summary_df[display_cols].copy()

    pd.set_option('display.max_columns', 20)
    pd.set_option('display.width', 280)
    pd.set_option('display.float_format', lambda x: f'{x:.4f}')
    print(f'\n{"="*90}')
    print('PU-Bagging Full Comparison (T=500)')
    print(f'{"="*90}')
    print(disp.to_string(index=False))
    print(f'{"="*90}')


### 5.1 Key Metrics Visualization

In [ ]:
if all_results:
    methods = disp['Feature Set'].values
    color_map = {
        'Raw Spectra 700-D': '#3498db',
        'CN-Aware AE-64d': '#f39c12',
        'Vanilla AE-64d': '#e74c3c',
        'AE-256d': '#2ecc71',
    }
    bar_colors = [color_map.get(m, '#95a5a6') for m in methods]

    metrics = [
        ('PR', 'PR-AUC (higher=better)'),
        ('Within-r', 'Within-Cluster r (higher=better)'),
        ('ROC', 'ROC-AUC (higher=better)'),
        ('P@100', 'P@100 (higher=better)'),
        ('|r_teff| z', '|r_teff| z-score (lower=better)'),
        ('Mean bias z', 'Mean Bias z (lower=better)'),
    ]

    fig, axes = plt.subplots(2, 3, figsize=(18, 9))
    axes = axes.flatten()

    for ax, (metric, title) in zip(axes, metrics):
        vals = disp[metric].values
        bars = ax.barh(range(len(methods)), vals, color=bar_colors, edgecolor='white', height=0.55)
        ax.set_yticks(range(len(methods)))
        ax.set_yticklabels([m[:22] for m in methods], fontsize=9)
        ax.set_title(title)
        ax.grid(axis='x', alpha=0.2)
        for bar, val in zip(bars, vals):
            offset = max(vals) * 0.02 if max(vals) > 0 else 0.005
            ax.text(bar.get_width() + offset, bar.get_y() + bar.get_height()/2,
                    f'{val:.4f}', va='center', fontsize=9, fontweight='bold')

    fig.suptitle('PU-Bagging: Complete Method Comparison (T=500)', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()


### 5.2 Probability Distribution Comparison

Compare how each method separates known CN stars from unlabeled stars
via PU-Bagging probability distributions.


In [ ]:
probs_paths = [
    ('CN-Aware + Vanilla + Spectra', Path('SpectraAE/results/cn_aware_pu_probs.csv')),
    ('AE-256d + AE-64d + Spectra', Path('SpectraAE/results/ae256_pu_probs.csv')),
    ('AE-64d + Spectra', Path('SpectraAE/results/ae_pu_probs.csv')),
]

probs_data = None
for label, fp in probs_paths:
    if fp.exists():
        probs_data = pd.read_csv(fp)
        print(f'Loaded probabilities from: {label}')
        print(f'  Columns: {list(probs_data.columns)}')
        print(f'  Rows: {len(probs_data)}')
        break

if probs_data is not None:
    prob_sets = []
    for col, name, color in [
        ('cn_aware_prob', 'CN-Aware AE-64d', '#f39c12'),
        ('ae256_prob', 'AE-256d', '#2ecc71'),
        ('ae64_prob', 'Vanilla AE-64d', '#e74c3c'),
        ('spec_prob', 'Raw Spectra 700-D', '#3498db'),
    ]:
        if col in probs_data.columns:
            prob_sets.append((name, probs_data[col].values, color))

    n_sets = len(prob_sets)
    if n_sets > 0:
        fig, axes = plt.subplots(1, n_sets, figsize=(5.5 * n_sets, 4.5))
        if n_sets == 1:
            axes = [axes]

        for ax, (name, probs, color) in zip(axes, prob_sets):
            ax.hist(probs[~pos_mask], bins=80, alpha=0.7, density=True, color='#bdc3c7',
                    label='Unlabeled')
            ax.hist(probs[pos_mask], bins=25, alpha=0.9, density=True, color=color,
                    label='Known CN')
            known_mean = probs[pos_mask].mean()
            ax.axvline(x=known_mean, color=color, linestyle='--', alpha=0.8, lw=1.5,
                      label=f'CN mean={known_mean:.3f}')
            ax.set_xlabel('PU-Bagging Probability')
            ax.set_ylabel('Density')
            ax.set_title(f'{name} (mean={probs.mean():.3f})')
            ax.legend(fontsize=7)
            ax.grid(alpha=0.2)

        fig.suptitle('PU-Bagging Probability Distributions: All Methods', fontsize=13, y=1.02)
        plt.tight_layout()
        plt.show()


### 5.3 CN-Aware vs Vanilla AE Probability Scatter

Analyze per-star PU-Bagging probability differences between CN-Aware and Vanilla AE,
identifying "divergent stars" where CN-Aware significantly outranks Vanilla.


In [ ]:
if probs_data is not None and 'cn_aware_prob' in probs_data.columns and 'ae64_prob' in probs_data.columns:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

    cn_prob = probs_data['cn_aware_prob'].values
    vn_prob = probs_data['ae64_prob'].values
    spec_prob = probs_data.get('spec_prob', None)
    if spec_prob is not None:
        spec_prob = spec_prob.values

    # Left: CN-Aware vs Vanilla
    ax = axes[0]
    ax.scatter(vn_prob[~pos_mask], cn_prob[~pos_mask], s=0.8, alpha=0.3,
              color='#95a5a6', edgecolors='none', label='Unlabeled')
    ax.scatter(vn_prob[pos_mask], cn_prob[pos_mask], s=30, alpha=0.9,
              color='#f39c12', edgecolors='white', linewidth=0.5, marker='*', label='Known CN')
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, lw=1)
    discordant = cn_prob - vn_prob
    top_discordant = np.argsort(discordant)[::-1][:20]
    ax.scatter(vn_prob[top_discordant], cn_prob[top_discordant], s=50, alpha=0.8,
              facecolors='none', edgecolors='red', linewidth=1.5, marker='o',
              label='Top 20 discordant')
    ax.set_xlabel('Vanilla AE-64d Probability')
    ax.set_ylabel('CN-Aware AE-64d Probability')
    ax.set_title(f'CN-Aware vs Vanilla AE (r={np.corrcoef(cn_prob, vn_prob)[0,1]:.3f})')
    ax.legend(fontsize=7, loc='lower right')
    ax.grid(alpha=0.15)

    # Right: CN-Aware vs Raw Spectra
    if spec_prob is not None:
        ax = axes[1]
        ax.scatter(spec_prob[~pos_mask], cn_prob[~pos_mask], s=0.8, alpha=0.3,
                  color='#95a5a6', edgecolors='none', label='Unlabeled')
        ax.scatter(spec_prob[pos_mask], cn_prob[pos_mask], s=30, alpha=0.9,
                  color='#f39c12', edgecolors='white', linewidth=0.5, marker='*', label='Known CN')
        ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, lw=1)
        ax.set_xlabel('Raw Spectra Probability')
        ax.set_ylabel('CN-Aware AE-64d Probability')
        ax.set_title(f'CN-Aware AE vs Raw Spectra (r={np.corrcoef(cn_prob, spec_prob)[0,1]:.3f})')
        ax.legend(fontsize=7, loc='lower right')
        ax.grid(alpha=0.15)

    fig.suptitle('PU-Bagging Probability Cross-Comparison', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

    print(f'\nTop 10 "CN-Aware >> Vanilla" divergent stars:')
    for i, idx in enumerate(top_discordant[:10]):
        label_str = 'CN+' if pos_mask[idx] else 'UNK'
        print(f'  #{i+1}: idx={idx}, CN-aware={cn_prob[idx]:.4f}, '
              f'Vanilla={vn_prob[idx]:.4f}, diff={discordant[idx]:.4f}, label={label_str}')
else:
    print('CN-Aware probability data not available')


## 6. Candidate Star Analysis

Based on PU-Bagging probabilities, select high-confidence CN star candidates,
analyze their spectral features and parameter space distribution.


In [ ]:
if probs_data is not None:
    cand_col = None
    for col in ['cn_aware_prob', 'ae256_prob', 'ae64_prob', 'spec_prob']:
        if col in probs_data.columns:
            cand_col = col
            break

    if cand_col:
        unl_mask = ~pos_mask
        cand_probs = probs_data[cand_col].values

        top_n = 30
        top_local = np.argsort(cand_probs[unl_mask])[::-1][:top_n]
        top_global = np.where(unl_mask)[0][top_local]

        candidates = pd.DataFrame({
            'rank': range(1, top_n + 1),
            'prob': cand_probs[top_global],
            'teff': df_model.iloc[top_global]['teff'].values,
            'logg': df_model.iloc[top_global]['logg'].values,
            'feh': df_model.iloc[top_global]['feh'].values,
        }).set_index('rank')

        for col in ['CN3839', 'CN4142', 'CH4300', 'delta_CN3839', 'delta_CN4142', 'delta_CH4300']:
            if col in df_model.columns:
                candidates[col] = df_model.iloc[top_global][col].values

        print(f'Top {top_n} candidates from {cand_col}:')
        display(candidates.head(15).style
            .background_gradient(subset=['prob'], cmap='Reds')
            .format('{:.4f}'))


### 6.1 Top 12 Candidate Spectra vs Cluster Median

Candidate spectra overlaid with their cluster median spectra,
showing absorption anomalies in CN band regions.


In [ ]:
n_cand_show = min(12, len(top_global) if 'top_global' in dir() else 12)

if 'top_global' in dir() and 'cand_probs' in dir():
    fig, axes = plt.subplots(n_cand_show, 1, figsize=(14, 2.2 * n_cand_show))
    if n_cand_show == 1:
        axes = [axes]

    for i in range(n_cand_show):
        ax = axes[i]
        g_idx = top_global[i]
        ax.plot(wave, X_clean[g_idx], lw=1.2, color='#e74c3c', label=f'Candidate #{i+1}')
        cid = cluster_ids[g_idx]
        if cid >= 0:
            cluster_members = np.where((cluster_ids == cid) & (~pos_mask))[0]
            if len(cluster_members) > 5:
                cluster_median = np.median(X_clean[cluster_members], axis=0)
                ax.plot(wave, cluster_median, lw=1, color='#3498db', alpha=0.7,
                       label=f'Cluster {cid} median')

        for _, (x1, x2), color in band_styles:
            ax.axvspan(x1, x2, alpha=0.1, color=color)

        ax.set_ylabel(f'#{i+1} (p={cand_probs[g_idx]:.3f})', fontsize=8)
        ax.legend(fontsize=7, loc='upper right')
        ax.grid(alpha=0.15)

    axes[-1].set_xlabel('Wavelength (A)')
    fig.suptitle(f'Top {n_cand_show} Candidates: {cand_col}', fontsize=13, y=1.005)
    plt.tight_layout()
    plt.show()


### 6.2 Teff-Logg Parameter Space Distribution

Candidate star positions on the HR diagram. CN-enhanced stars typically
occupy specific Teff-logg regions.


In [ ]:
if 'top_global' in dir():
    fig, ax = plt.subplots(figsize=(10, 8))

    unl_teff = df_model.iloc[~pos_mask]['teff'].values
    unl_logg = df_model.iloc[~pos_mask]['logg'].values
    hb = ax.hexbin(unl_teff, unl_logg, gridsize=80, cmap='Greys', mincnt=1, alpha=0.6)
    plt.colorbar(hb, ax=ax, label='Star count', shrink=0.8)

    cn_teff = df_model.iloc[pos_mask]['teff'].values
    cn_logg = df_model.iloc[pos_mask]['logg'].values
    ax.scatter(cn_teff, cn_logg, s=80, alpha=0.9, c='#e74c3c', edgecolors='white',
              linewidth=1, marker='*', zorder=5, label=f'Known CN (n={n_pos})')

    cand_teff = df_model.iloc[top_global]['teff'].values
    cand_logg = df_model.iloc[top_global]['logg'].values
    probs_top = cand_probs[top_global]
    sc = ax.scatter(cand_teff, cand_logg, s=80, alpha=0.85, c=probs_top,
                   cmap='YlOrRd', edgecolors='black', linewidth=0.8,
                   marker='D', zorder=6, label=f'Top {len(top_global)} Candidates')
    plt.colorbar(sc, ax=ax, label='PU-Bagging Probability', shrink=0.8)

    ax.set_xlabel('Teff (K)')
    ax.set_ylabel('log g')
    ax.set_title('Candidate CN Stars in Teff-Logg Space')
    ax.invert_xaxis()
    ax.invert_yaxis()
    ax.legend(fontsize=9, loc='upper right')
    ax.grid(alpha=0.15)

    plt.tight_layout()
    plt.show()


## 7. Summary and Conclusions

### Performance Ranking (by PR-AUC descending)

| Method | Dim | PR-AUC | ROC | P@100 | Within-r | |r_teff| z | Time |
|--------|-----|--------|-----|-------|----------|------------|------|
| **Raw Spectra + PU-Bagging** | 700 | **0.848** | **0.997** | **0.10** | 0.244 | 0.043 | ~279s |
| **CN-Aware AE-64d + PU-Bagging** | 64 | **0.181** | 0.972 | 0.07 | **0.279** | 0.032 | ~44s |
| AE-256d + PU-Bagging | 256 | 0.173 | 0.923 | 0.05 | 0.279 | 0.019 | ~112s |
| Vanilla AE-64d + PU-Bagging | 64 | 0.167 | 0.918 | 0.04 | 0.236 | **0.007** | ~39s |

### Core Findings

**1. Standard MSE autoencoding does NOT preserve CN-discriminative information**

AE bottleneck features achieve PR-AUC of only ~0.17, far below raw spectra 0.85. Reasons:
- Continuum shape dominates ~99% of pixel variance
- CN molecular band variations account for only ~0.5% of variance
- Standard MSE loss has no mechanism to prioritize these weak but scientifically critical signals

**2. CN-Aware training provides modest improvement**

5x band-weighted CN-Aware AE achieves PR=0.181 (vs vanilla 0.167), ~8.4% relative improvement:
- Higher ROC-AUC (0.972 vs 0.918)
- Higher within-cluster correlation (Within-r: 0.279 vs 0.236) -- better physical consistency
- Higher P@100 (0.07 vs 0.04) -- finding more high-confidence candidates

However, improvement is limited. Weighted MSE is still fundamentally a reconstruction-oriented objective,
unable to fundamentally transform the information content of bottleneck features.

**3. Larger bottleneck dimension helps marginally**

256d vs 64d shows only slight improvement (PR 0.173 vs 0.167), while tripling training time.
Extra capacity primarily captures more continuum detail rather than CN-specific features.

**4. AE features have unique complementary value**

Although PR-AUC is low, AE features excel in certain aspects:
- CN-Aware AE has highest Within-r (0.279), indicating more physically consistent ranking
- CN-Aware AE has lower Teff bias (|r_teff| z = 0.032 vs Raw Spectra 0.043)
- This suggests AE bottleneck features, while weak in overall discrimination, may contain information complementary to raw spectra


### 7.1 AE Variants Radar Chart Comparison

In [ ]:
if all_results:
    radar_data = disp[disp['Feature Set'] != 'Raw Spectra 700-D'].copy()

    categories = ['PR-AUC', 'ROC-AUC', 'P@100', 'Within-r', '1-|r_teff| z', 'Speed']
    n_cat = len(categories)
    angles = np.linspace(0, 2 * np.pi, n_cat, endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

    colors_radar = {'CN-Aware AE-64d': '#f39c12', 'Vanilla AE-64d': '#e74c3c', 'AE-256d': '#2ecc71'}

    for _, row in radar_data.iterrows():
        name = row['Feature Set']
        vals = [
            row['PR'] / radar_data['PR'].max(),
            row['ROC'] / radar_data['ROC'].max(),
            row['P@100'] / radar_data['P@100'].max(),
            row['Within-r'] / radar_data['Within-r'].max(),
            1 - row['|r_teff| z'],
            1 / row['Time'] * radar_data['Time'].min(),
        ]
        vals += vals[:1]
        ax.fill(angles, vals, alpha=0.1, color=colors_radar.get(name, '#95a5a6'))
        ax.plot(angles, vals, 'o-', lw=2, color=colors_radar.get(name, '#95a5a6'), label=name, markersize=6)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=11)
    ax.set_yticklabels([])
    ax.set_title('AE Variants Comparison (normalized, excl. Raw Spectra)', fontsize=13, pad=25)
    ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=10)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


## 8. Next Steps and Recommendations

The fundamental limitation of autoencoders (including CN-Aware variants) is:
**there is an inherent mismatch between unsupervised MSE reconstruction and CN classification objectives**.
Continuum shape dominates variance, and MSE optimization naturally ignores molecular band signals
that account for only ~0.5% of variance.

Based on experimental results, recommended priority order:

### Short-term (1-2 days, low risk)

**Strategy B: PU-Learning + Deep Neural Network**
- Directly replace XGBoost with ResNet/DNN in PU-Bagging framework
- Use data augmentation (noise, RV jitter, continuum tilt) to expand positive samples
- Expected PR: 0.3-0.5
- Existing infrastructure: Deep/models/resnet1d.py, Deep/pu_improved.py

### Medium-term (1 week, medium risk)

**Strategy A: Masked Spectrum Modeling (MSM)** [RECOMMENDED]
- MAE/BERT paradigm applied to 1D spectra: randomly mask 30-50% pixels, train encoder to reconstruct
- Purely unsupervised, utilizes all 33,589 spectra
- Lower mask rate in CN band regions (10%) to preserve key information
- After pretraining, freeze encoder + lightweight classification head fine-tuning
- Expected PR: 0.4-0.7
- This is the most promising path to surpass XGBoost

**Strategy D: Knowledge Distillation + Pseudo-labeling**
- Use Raw Spectra PU-Bagging (PR=0.848) as teacher model
- Select 100-200 high-confidence (prob>0.7) pseudo-labels
- Train deep student model with self-training iterations
- Expected PR: 0.5-0.7
- Risk: pseudo-label bias propagates to student model

### Long-term (1 week+, high risk high reward)

**Strategy C: Contrastive Learning**
- SimCLR style: different augmented versions of same spectrum as positive pairs
- Augmentations: Gaussian noise, wavelength shift, local continuum scaling, random pixel masking
- NT-Xent loss, requires large batch size (512+)
- Expected PR: 0.3-0.6
- Risk: negative sampling strategy critically affects results, needs careful tuning

### Key Considerations

1. **Unified evaluation**: Always use the same 85/15 stratified split for XGBoost baseline comparability
2. **Overfitting prevention**: Only 58 training positives, use validation PR for early stopping
3. **Bias checking**: After each training run, check |r_teff| z and Within-r to ensure model learns CN signal, not temperature bias
4. **Progressive validation**: Test on CN 9-D features first (fast), then extend to full spectra
5. **Feature fusion**: Although AE bottleneck features have weak discrimination, they may complement raw spectra -- try multi-modal fusion

---
*Notebook generated: 2026-05-28 | Data: 33,589 LAMOST spectra | 73 known CN stars*
